# Segmentasi gigi + penomoran FDI (permanen & susu)

Pengganti `teeth_segmentation_sam2.ipynb`. Backend segmentasi **bisa ditukar** lewat
`CFG["backend"]`; default **SAM 2** karena SAM 3 belum stabil di environment ini.

## Dua kegagalan yang diperbaiki

Diagnosis di Section 10 notebook sebelumnya menunjukkan pipeline lama gagal total pada
sebagian foto:

```
SAM 2 menghasilkan 6 mask MENTAH untuk seluruh gambar
Lolos filter: 0 dari 6   — semuanya tertolak "saturasi > 0.45"
```

**Masalah 1 — parameter AMG terlalu ketat.** Default HuggingFace (`pred_iou_thresh` ~0.88,
`stability_score_thresh` ~0.95, grid 32×32) dibuat untuk foto natural berkontras tinggi. Pada
foto intraoral, ambang itu membunuh hampir semua kandidat. Di notebook ini dilonggarkan ke
0.70 / 0.80, grid 48×48, plus `crops_n_layers=1`.

**Masalah 2 — filter warna absolut.** `sat_max = 0.45` mengasumsikan "gigi putih, gusi merah".
Rongga mulut berselubung pink **seluruhnya**, dan gigi memantulkannya — saturasi gigi terukur
0.49–0.56, jadi semuanya tertolak. Sekarang ambangnya **adaptif**: gigi adalah objek paling
tidak jenuh di dalam mulut, jadi ambang diambil dari persentil distribusi saturasi foto itu
sendiri.

Ini pola yang sama dengan dua bug sebelumnya (`edge_density`, `rough_mean`): **ambang absolut
pada besaran yang skalanya berbeda tiap foto.**

## Kalau SAM 3 sudah jalan

Set `CFG["backend"]="sam3"`. SAM 3 melakukan *Promptable Concept Segmentation* — diberi teks
`"tooth"` ia mengembalikan mask setiap instance yang cocok, sehingga **filter warna tidak
diperlukan sama sekali** (kode otomatis melewatinya). Itu jalur yang lebih bersih; sisa notebook
ini tidak berubah sedikit pun. Kalau `Sam3Processor` error, biasanya `transformers` perlu
di-upgrade: `pip install -U "transformers[torch]"`.

## Perbandingan dengan notebook sebelumnya

| | `teeth_segmentation_sam2.ipynb` | Notebook ini |
|---|---|---|
| Parameter AMG | default HF (ketat) | **dilonggarkan** + `crops_n_layers=1` |
| Filter warna | `sat_max=0.45` absolut | **persentil adaptif per foto** |
| Midline | median sentroid-x | pasangan bertetangga **terlebar & paling setara** |
| Penomoran FDI | peringkat dari midline | **DP alignment** (Needleman–Wunsch) terhadap template lebar |
| Gigi hilang | menggeser semua nomor diam-diam | ditangani sebagai **gap** dalam alignment |
| Gigi susu | tidak didukung | **didukung** — kuadran 5–8 |
| Backend | SAM 2 saja | SAM 2 / SAM 3 / auto |

Penomoran berbasis peringkat punya cacat fatal: satu gigi gagal tersegmentasi, semua nomor
sesudahnya bergeser tanpa error. DP alignment menyisipkan gap alih-alih menggeser, jadi kegagalan
menjadi terlihat, bukan senyap.

## 0. Notasi FDI — permanen dan susu

FDI dua digit: **digit pertama = kuadran**, **digit kedua = posisi dari midline**.

| Kuadran | Sisi (pasien) | Permanen | Susu |
|---|---|---|---|
| Kanan atas | kiri gambar | **1**1–18 | **5**1–55 |
| Kiri atas | kanan gambar | **2**1–28 | **6**1–65 |
| Kiri bawah | kanan gambar | **3**1–38 | **7**1–75 |
| Kanan bawah | kiri gambar | **4**1–48 | **8**1–85 |

Permanen punya 8 gigi per kuadran, susu hanya 5 (dua insisivus, satu kaninus, dua molar).
Karena foto frontal, yang realistis terlihat hanya posisi 1–3 (kadang 4).

**Ingat: kiri gambar = sisi KANAN pasien.**

### Membedakan gigi susu dari permanen

Tiga kriteria, semuanya terhitung dari mask:

1. **Rasio lebar/tinggi mahkota.** Ini yang paling tegas dan bisa disitasi: pada insisivus sentral
   **susu**, diameter mesiodistal *lebih besar* dari panjang cervicoincisal (**w/h > 1**);
   pada insisivus sentral **permanen**, kebalikannya (**w/h < 1**).
2. **Ukuran relatif.** Gigi anterior susu lebih kecil dari penggantinya yang permanen.
3. **Warna.** Gigi susu lebih terang dengan semburat kebiruan; gigi permanen lebih kuning/abu.

Ketiganya digabung jadi satu skor, dan **geometri boleh menimpanya** saat DP alignment berjalan —
kalau lebar sebuah gigi jauh lebih cocok sebagai permanen, tebakan awal dikoreksi.

## 1. Instalasi

```bash
pip install -U "transformers[torch]"
```

Default memakai `facebook/sam2.1-hiera-large` (~900 MB, tidak gated). Untuk mencoba SAM 3,
set `CFG["backend"]="sam3"` — bobot `facebook/sam3` (~3 GB) akan terunduh otomatis, tapi ia
butuh `transformers` versi baru.

`CFG["backend"]="auto"` mencoba SAM 3 dulu lalu mundur ke SAM 2 kalau gagal — berguna kalau
kamu ingin notebook tetap jalan tanpa memutuskan lebih dulu.

Catatan MPS: kalau kehabisan memori, set `CFG["force_cpu"]=True` atau turunkan
`CFG["max_side"]`. `crops_n_layers=1` menggandakan waktu inferensi tapi menaikkan jumlah
mask secara signifikan — turunkan ke 0 kalau terlalu lambat.

In [ ]:
# ========= CONFIG =========
CFG = {
    "img_dir":   "Front Teeth drg Laura",
    "out_dir":   "seg3_out",
    "force_cpu": False,
    "max_side":  1024,

    # --- backend segmentasi: "sam2" | "sam3" | "auto" (coba sam3 -> mundur ke sam2) ---
    "backend":   "sam2",
    "sam3_id":   "facebook/sam3",
    "sam2_id":   "facebook/sam2.1-hiera-large",

    # --- SAM 3 (prompt teks) ---
    "text_prompt":    "tooth",
    "score_thresh":   0.35,
    "mask_thresh":    0.50,

    # --- SAM 2 (automatic mask generation) — DILONGGARKAN dari default ---
    # Default HF terlalu ketat utk foto intraoral berkontras rendah: pada run sebelumnya
    # SAM 2 cuma menghasilkan 6 mask untuk SELURUH gambar.
    "points_per_crop":        48,     # default 32 -> grid lebih rapat
    "pred_iou_thresh":        0.70,   # default ~0.88
    "stability_score_thresh": 0.80,   # default ~0.95
    "crops_n_layers":         1,      # pass kedua pada crop -> objek kecil ikut terjaring
    "points_per_batch":       64,

    # --- filter warna: hanya dipakai jalur SAM 2, dan ADAPTIF ---
    # Ambang absolut (sat_max=0.45) menolak SEMUA mask di sebagian foto: rongga mulut
    # berselubung pink seluruhnya. Gigi adalah objek PALING TIDAK JENUH di dalam mulut,
    # jadi ambangnya diambil dari distribusi saturasi foto itu sendiri.
    "sat_pct":        35,     # persentil saturasi gambar sbg ambang atas
    "val_pct":        55,     # persentil value gambar sbg ambang bawah
    "solidity_min":   0.70,

    "nms_iou":        0.55,
    "min_area_frac":  0.0015,
    "max_area_frac":  0.12,
    "max_teeth":      20,

    # --- klasifikasi susu vs permanen ---
    "wh_split":       1.00,     # w/h > 1 -> cenderung susu (utk insisivus)
    "type_penalty":   0.35,     # penalti kalau DP tidak setuju dgn tebakan awal

    # --- DP alignment ---
    "w_weight":       1.00,     # bobot bukti LEBAR
    "pos_weight":     1.00,     # bobot bukti POSISI (jarak dari midline) — penting utk gigi hilang
    "gap_template":   0.55,     # biaya slot FDI kosong (gigi hilang / tak tersegmentasi)
    "gap_observed":   0.85,     # biaya objek terdeteksi yg tak masuk template (positif palsu)
    "n_slots":        4,        # slot per sisi yang dicoba (1..4)

    # --- landmark ---
    "contact_band": (0.35, 0.90),
    "incisal_frac": 0.12,
}
import os, json
os.makedirs(CFG["out_dir"], exist_ok=True)
print(json.dumps(CFG, indent=1))

## 2. Segmentasi dengan prompt teks

In [ ]:
import glob, time, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as MplPoly
from PIL import Image
import torch
import warnings; warnings.filterwarnings("ignore")
try:
    from pillow_heif import register_heif_opener; register_heif_opener()
except Exception: pass

DEV = torch.device("cpu") if CFG["force_cpu"] else torch.device(
    "mps" if torch.backends.mps.is_available() else
    ("cuda" if torch.cuda.is_available() else "cpu"))
print("Device:", DEV)

EXTS = ("*.jpg","*.jpeg","*.JPG","*.JPEG","*.png","*.PNG","*.heic","*.HEIC")
paths = sorted(set(sum([glob.glob(os.path.join(CFG["img_dir"], e)) for e in EXTS], [])))
names = [os.path.splitext(os.path.basename(p))[0] for p in paths]
print(f"{len(paths)} foto")

def load_rgb(p):
    im = Image.open(p).convert("RGB")
    if max(im.size) > CFG["max_side"]:
        sc = CFG["max_side"]/max(im.size)
        im = im.resize((int(im.width*sc), int(im.height*sc)), Image.LANCZOS)
    return im

In [ ]:
import transformers
print("transformers:", transformers.__version__)

BACKEND, _sam3, _sam2 = None, None, None

def _load_sam3():
    from transformers import Sam3Model, Sam3Processor
    m = Sam3Model.from_pretrained(CFG["sam3_id"]).to(DEV).eval()
    p = Sam3Processor.from_pretrained(CFG["sam3_id"])
    return m, p

def _load_sam2():
    from transformers import pipeline
    dev = 0 if DEV.type == "cuda" else (DEV.type if DEV.type == "mps" else -1)
    return pipeline("mask-generation", model=CFG["sam2_id"], device=dev)

order = {"auto": ["sam3","sam2"], "sam3": ["sam3"], "sam2": ["sam2"]}[CFG["backend"]]
for b in order:
    try:
        if b == "sam3": _sam3 = _load_sam3()
        else:           _sam2 = _load_sam2()
        BACKEND = b; print(f"Backend aktif: {b}"); break
    except Exception as e:
        print(f"  {b} gagal dimuat: {type(e).__name__}: {str(e)[:160]}")
if BACKEND is None:
    raise RuntimeError("Tidak ada backend yang bisa dimuat. Coba: pip install -U \"transformers[torch]\"")

@torch.no_grad()
def _segment_sam3(pil):
    model, processor = _sam3
    inputs = processor(images=pil, text=CFG["text_prompt"], return_tensors="pt").to(DEV)
    res = processor.post_process_instance_segmentation(
        model(**inputs), threshold=CFG["score_thresh"], mask_threshold=CFG["mask_thresh"],
        target_sizes=inputs.get("original_sizes").tolist())[0]
    out = [(np.asarray(m, bool), float(s)) for m, s in zip(res["masks"], res["scores"])]
    return sorted(out, key=lambda t: -t[1])

def _segment_sam2(pil):
    out = _sam2(pil,
                points_per_batch=CFG["points_per_batch"],
                points_per_crop=CFG["points_per_crop"],
                crops_n_layers=CFG["crops_n_layers"],
                pred_iou_thresh=CFG["pred_iou_thresh"],
                stability_score_thresh=CFG["stability_score_thresh"])
    masks = [np.asarray(m, bool) for m in out["masks"]]
    scores = [float(s) for s in out.get("scores", [1.0]*len(masks))]
    return sorted(zip(masks, scores), key=lambda t: -t[1])

def segment(pil):
    """-> list of (mask bool HxW, score). Seragam untuk kedua backend."""
    return _segment_sam3(pil) if BACKEND == "sam3" else _segment_sam2(pil)

# uji satu foto lebih dulu — kegagalan ketahuan sekarang, bukan 20 menit lagi
_t0 = time.time()
_test = segment(load_rgb(paths[0]))
print(f"Uji '{names[0][:40]}': {len(_test)} mask mentah dalam {time.time()-_t0:.1f}s")
if BACKEND == "sam2" and len(_test) < 15:
    print("  PERINGATAN: mask mentah sedikit. Longgarkan lagi pred_iou_thresh /")
    print("  stability_score_thresh, atau naikkan points_per_crop di CFG.")

In [ ]:
def iou(a, b):
    inter = np.logical_and(a, b).sum()
    return 0.0 if inter == 0 else float(inter/np.logical_or(a, b).sum())

def mask_geom(m):
    """
    w = lebar MESIODISTAL, h = tinggi CERVICOINCISAL.

    PENTING: sumbu TIDAK boleh dipilih berdasarkan mana yang lebih panjang. Insisivus sentral
    SUSU justru lebih lebar daripada tinggi (w/h > 1) — itu kriteria pembedanya. Memaksa sumbu
    panjang selalu jadi 'h' akan membuat w/h tak pernah melebihi 1 dan membunuh sinyalnya.
    Karena itu sumbu dipilih berdasarkan ORIENTASI: yang lebih dekat ke vertikal = cervicoincisal.
    """
    ys, xs = np.nonzero(m)
    if len(xs) < 25: return None
    pts = np.stack([xs, ys], 1).astype(float)
    c = pts.mean(0); X = pts - c
    ev, evec = np.linalg.eigh(X.T @ X / len(X))
    e0, e1 = evec[:, 0], evec[:, 1]
    vert = e0 if abs(e0[1]) >= abs(e1[1]) else e1      # sumbu paling dekat ke vertikal
    horiz = e1 if vert is e0 else e0
    h = float(np.ptp(X @ vert)); w = float(np.ptp(X @ horiz))
    ang = float(np.degrees(np.arctan2(vert[0], vert[1])))
    if ang > 90: ang -= 180
    if ang <= -90: ang += 180
    return {"cx": c[0], "cy": c[1], "w": w, "h": h, "tilt": ang,
            "area": float(m.sum()), "wh": w/max(h, 1e-6),
            "y0": float(ys.min()), "y1": float(ys.max())}

def convex_area(mask):
    ys, xs = np.nonzero(mask)
    if len(xs) < 3: return float(mask.sum())
    pts = sorted(set(zip(xs.tolist(), ys.tolist())))
    if len(pts) < 3: return float(mask.sum())
    cross = lambda o,a,b: (a[0]-o[0])*(b[1]-o[1]) - (a[1]-o[1])*(b[0]-o[0])
    lo = []
    for p in pts:
        while len(lo) >= 2 and cross(lo[-2], lo[-1], p) <= 0: lo.pop()
        lo.append(p)
    up = []
    for p in reversed(pts):
        while len(up) >= 2 and cross(up[-2], up[-1], p) <= 0: up.pop()
        up.append(p)
    hull = lo[:-1] + up[:-1]
    if len(hull) < 3: return float(mask.sum())
    a = sum(hull[i][0]*hull[(i+1)%len(hull)][1] - hull[(i+1)%len(hull)][0]*hull[i][1]
            for i in range(len(hull)))
    return abs(a)/2 + 1e-6

def clean(instances, rgb):
    """
    NMS + saring luas, plus filter warna ADAPTIF (hanya untuk jalur SAM 2).

    SAM 3 tidak butuh filter warna: konsep "tooth" yang menyaring. SAM 2 memakai automatic
    mask generation dan mengembalikan gusi/bibir/lidah juga, jadi filter tetap perlu — tapi
    ambangnya diambil dari distribusi foto itu sendiri, bukan konstanta. Versi sebelumnya
    memakai sat_max=0.45 absolut dan menolak SEMUA mask di sebagian foto.
    """
    H, W = rgb.shape[:2]; A = H*W
    rgb01 = rgb.astype(np.float32)/255.
    mx = rgb01.max(-1); mn = rgb01.min(-1)
    sat_img = np.where(mx > 1e-6, (mx-mn)/(mx+1e-6), 0.0)
    need_color = (BACKEND == "sam2")
    sat_thr = float(np.percentile(sat_img, CFG["sat_pct"]))   # gigi = paling tidak jenuh
    val_thr = float(np.percentile(mx,      CFG["val_pct"]))
    keep, rejected = [], {"luas": 0, "warna": 0, "soliditas": 0, "duplikat": 0}
    for m, s in instances:
        g = mask_geom(m)
        if g is None: continue
        if not (CFG["min_area_frac"] <= g["area"]/A <= CFG["max_area_frac"]):
            rejected["luas"] += 1; continue
        px = rgb01[m]
        g["sat"] = float(sat_img[m].mean())
        g["val"] = float(px.max(-1).mean()) if len(px) else 0.0
        g["yellow"] = float((((px[:,0]+px[:,1])/2 - px[:,2])).mean()) if len(px) else 0.0
        if need_color and (g["sat"] > sat_thr or g["val"] < val_thr):
            rejected["warna"] += 1; continue
        g["solidity"] = g["area"]/convex_area(m)
        if need_color and g["solidity"] < CFG["solidity_min"]:
            rejected["soliditas"] += 1; continue
        if any(iou(m, k[0]) > CFG["nms_iou"] for k in keep):
            rejected["duplikat"] += 1; continue
        g["score"] = s
        keep.append((m, g))
        if len(keep) >= CFG["max_teeth"]: break
    return keep, {"sat_thr": sat_thr, "val_thr": val_thr, **rejected}

SEG, DIAG = {}, {}
t0 = time.time()
for p, n in zip(paths, names):
    try:
        rgb = np.asarray(load_rgb(p), np.uint8)
        raw = segment(Image.fromarray(rgb))
        keep, info = clean(raw, rgb)
        SEG[n] = {"rgb": rgb, "keep": keep}; DIAG[n] = {"mentah": len(raw), **info}
        flag = "  <-- sedikit" if len(keep) < 8 else ""
        print(f"  {n[:42]:42s} mentah {len(raw):3d} -> {len(keep):2d} gigi"
              f"  (tolak: luas {info['luas']}, warna {info['warna']}, "
              f"sol {info['soliditas']}, dup {info['duplikat']}){flag}")
    except Exception as e:
        print(f"  {n[:42]:42s} GAGAL: {type(e).__name__} {str(e)[:70]}")
print(f"\n{len(SEG)} foto dalam {(time.time()-t0)/60:.1f} menit")
if DIAG:
    D = pd.DataFrame(DIAG).T
    print(f"\nrata-rata: {D['mentah'].mean():.0f} mask mentah -> "
          f"{np.mean([len(SEG[n]['keep']) for n in SEG]):.1f} gigi")
    print("Kalau 'warna' jadi penolak terbesar, naikkan sat_pct / turunkan val_pct di CFG.")

## 3. Pisah lengkung & jangkar midline

**Midline tidak lagi diambil dari median sentroid** — itu bergeser begitu ada diastema atau gigi
hilang. Sebagai gantinya: cari **pasangan gigi bertetangga yang memaksimalkan**

```
skor = (w_a + w_b) − λ·|w_a − w_b|
```

yaitu pasangan terlebar yang juga paling setara lebarnya. Itulah insisivus sentral. Kriteria ini
bertahan meski lateral atau kaninus hilang, dan bertahan meski ada diastema di antara keduanya.

In [ ]:
def split_arches(keep):
    cy = np.array([g["cy"] for _, g in keep], float)
    if len(cy) < 2: return list(range(len(cy))), []
    c = np.array([cy.min(), cy.max()], float)
    for _ in range(60):
        lab = np.abs(cy[:, None] - c[None, :]).argmin(1)
        for k in (0, 1):
            if (lab == k).any(): c[k] = cy[lab == k].mean()
    up = [i for i in range(len(cy)) if lab[i] == 0]
    lo = [i for i in range(len(cy)) if lab[i] == 1]
    return (up, lo) if c[0] <= c[1] else (lo, up)

def central_pair(keep, idxs, lam=0.8):
    """Pasangan bertetangga terlebar & paling setara -> (i_kiri, i_kanan, x_midline)."""
    if len(idxs) < 2: return None
    order = sorted(idxs, key=lambda i: keep[i][1]["cx"])
    best, bs = None, -1e9
    for a, b in zip(order[:-1], order[1:]):
        wa, wb = keep[a][1]["w"], keep[b][1]["w"]
        s = (wa + wb) - lam*abs(wa - wb)
        if s > bs: bs, best = s, (a, b)
    a, b = best
    mid = 0.5*(keep[a][1]["cx"] + keep[b][1]["cx"])
    return a, b, mid

## 4. Susu atau permanen?

Skor gabungan tiga bukti. Nilai **positif = cenderung susu**, negatif = cenderung permanen.

| Bukti | Arah |
|---|---|
| `wh` (lebar/tinggi) | > 1.00 → susu. Kriteria paling tegas dan bisa disitasi |
| ukuran relatif | jauh lebih kecil dari gigi terlebar di lengkung → susu |
| warna | lebih terang & kurang kuning → susu |

Ini **tebakan awal**, bukan keputusan final. DP alignment di langkah berikutnya boleh menimpanya
kalau lebar gigi jauh lebih cocok sebagai tipe yang lain — geometri lebih dipercaya daripada warna.

In [ ]:
def primary_score(g, ref_w, ref_yellow):
    """> 0 cenderung SUSU, < 0 cenderung PERMANEN."""
    s_wh   = np.tanh((g["wh"] - CFG["wh_split"]) * 4.0)          # bukti terkuat
    s_size = np.tanh((0.85 - g["w"]/max(ref_w, 1e-6)) * 3.0)      # lebih kecil -> susu
    s_col  = np.tanh((ref_yellow - g["yellow"]) * 12.0)           # kurang kuning -> susu
    return float(0.55*s_wh + 0.30*s_size + 0.15*s_col)

def arch_types(keep, idxs):
    if not idxs: return {}
    ref_w = max(keep[i][1]["w"] for i in idxs)
    ref_y = float(np.median([keep[i][1]["yellow"] for i in idxs]))
    return {i: primary_score(keep[i][1], ref_w, ref_y) for i in idxs}

## 5. Penomoran FDI lewat DP alignment

Gigi terdeteksi pada satu sisi adalah **barisan** dari midline ke luar. Template FDI juga barisan.
Mencocokkan dua barisan yang salah satunya boleh punya lubang adalah **Needleman–Wunsch**.

```
biaya_cocok(i, j, t) = | log(w_i / w_ref) − log(r[j][t]) | + μ · tidak_setuju(t, tebakan_i)
biaya_gap_template   = λ_miss     (slot FDI kosong  → gigi hilang / tak tersegmentasi)
biaya_gap_observasi  = λ_extra    (objek tak masuk template → positif palsu)
```

`t ∈ {permanen, susu}` dipilih per slot, sehingga **dentisi campuran tertangani secara alami** —
insisivus sentral permanen boleh berdampingan dengan lateral susu.

**Skala `w_ref`.** Semua rasio relatif terhadap lebar insisivus sentral permanen. Tapi kalau
sentral yang terlihat justru gigi susu, skalanya berbeda. Solusinya: jalankan DP di bawah **dua
hipotesis** (sentral permanen / sentral susu), lalu ambil yang total biayanya lebih rendah. Ini
sekaligus menentukan tahap dentisi lengkung itu.

Rasio lebar mesiodistal dinormalisasi terhadap insisivus sentral permanen = 1.00, dari nilai
tekstbook (Wheeler/Ash). **Rasio jauh lebih stabil antar populasi daripada milimeter absolut**,
tapi sel di Bagian 8 tetap menyediakan cara membangun ulang template dari datamu sendiri.

In [ ]:
# rasio lebar mesiodistal, dinormalisasi ke insisivus sentral PERMANEN = 1.00
# slot 1..4 dari midline ke luar
RATIO = {
  "upper": {"perm": {1: 1.00, 2: 0.76, 3: 0.89, 4: 0.84},   # sentral, lateral, kaninus, P1
            "prim": {1: 0.76, 2: 0.60, 3: 0.82, 4: 0.86}},  # sentral, lateral, kaninus, M1 susu
  "lower": {"perm": {1: 0.62, 2: 0.67, 3: 0.80, 4: 0.82},
            "prim": {1: 0.49, 2: 0.54, 3: 0.59, 4: 0.91}},
}
QUAD = {("upper","left","perm"):1, ("upper","right","perm"):2,
        ("lower","right","perm"):3, ("lower","left","perm"):4,
        ("upper","left","prim"):5, ("upper","right","prim"):6,
        ("lower","right","prim"):7, ("lower","left","prim"):8}

def _expected_dist(R, t, S):
    """Jarak pusat gigi slot j dari midline, dlm satuan lebar sentral permanen."""
    D, cum = {}, 0.0
    for j in range(1, S+1):
        r = R[t].get(j)
        if r is None: break
        D[j] = cum + r/2; cum += r
    return D

def align_side(widths, dists, ptypes, w_ref, arch):
    """
    Needleman-Wunsch: cocokkan gigi (dari midline ke luar) ke slot 1..n_slots.
    -> (total_cost, assign) dengan assign[i] = (slot, tipe) atau None (positif palsu).

    Biaya memakai DUA bukti. Lebar saja tidak cukup: selisih lateral vs kaninus hanya ~17%,
    jadi kalau sebuah gigi gagal tersegmentasi, DP lebih murah menggeser nomor daripada
    menyisipkan gap. Jarak dari midline yang membuat gigi hilang benar-benar terdeteksi —
    gigi berikutnya duduk jauh lebih ke luar daripada yang diprediksi slotnya.
    """
    n, S = len(widths), CFG["n_slots"]
    R = RATIO[arch]
    ED = {t: _expected_dist(R, t, S) for t in ("perm", "prim")}
    def cost(i, j):
        best, bt = 1e9, "perm"
        for t in ("perm", "prim"):
            if j not in R[t]: continue
            c  = CFG["w_weight"]  * abs(np.log(max(widths[i], 1e-6)/w_ref) - np.log(R[t][j]))
            c += CFG["pos_weight"] * abs(dists[i]/w_ref - ED[t][j])
            c += CFG["type_penalty"] * (1.0 if ((t == "prim") != (ptypes[i] > 0)) else 0.0)
            if c < best: best, bt = c, t
        return best, bt

    INF = 1e12
    D = np.full((n+1, S+1), INF); D[0,0] = 0.0
    BP = {}
    for i in range(n+1):
        for j in range(S+1):
            if D[i,j] >= INF: continue
            if i < n and j < S:                                  # cocokkan
                c, t = cost(i, j+1)
                if D[i,j]+c < D[i+1,j+1]: D[i+1,j+1] = D[i,j]+c; BP[(i+1,j+1)] = (i,j,"M",j+1,t)
            if j < S:                                            # slot kosong
                if D[i,j]+CFG["gap_template"] < D[i,j+1]:
                    D[i,j+1] = D[i,j]+CFG["gap_template"]; BP[(i,j+1)] = (i,j,"T",None,None)
            if i < n:                                            # deteksi berlebih
                if D[i,j]+CFG["gap_observed"] < D[i+1,j]:
                    D[i+1,j] = D[i,j]+CFG["gap_observed"]; BP[(i+1,j)] = (i,j,"O",None,None)
    j_end = int(np.argmin(D[n, :])); total = float(D[n, j_end])
    assign = [None]*n
    i, j = n, j_end
    while (i, j) in BP:
        pi, pj, kind, slot, t = BP[(i,j)]
        if kind == "M": assign[pi] = (slot, t)
        i, j = pi, pj
    return total, assign

def number_arch(keep, idxs, tscore, arch):
    """-> {index_mask: nomor_FDI}, info hipotesis terpilih."""
    cp = central_pair(keep, idxs)
    if cp is None: return {}, None
    ia, ib, mid = cp
    w_c = 0.5*(keep[ia][1]["w"] + keep[ib][1]["w"])
    left  = sorted([i for i in idxs if keep[i][1]["cx"] <  mid], key=lambda i: -keep[i][1]["cx"])
    right = sorted([i for i in idxs if keep[i][1]["cx"] >= mid], key=lambda i:  keep[i][1]["cx"])

    best = None
    for hypo, r1 in (("perm", RATIO[arch]["perm"][1]), ("prim", RATIO[arch]["prim"][1])):
        w_ref = w_c / r1                      # skala "setara sentral permanen"
        tot, res = 0.0, {}
        for side, seq in (("left", left), ("right", right)):
            if not seq: continue
            c, a = align_side([keep[i][1]["w"] for i in seq],
                              [abs(keep[i][1]["cx"] - mid) for i in seq],
                              [tscore.get(i, 0.0) for i in seq], w_ref, arch)
            tot += c; res[side] = (seq, a)
        if best is None or tot < best[0]: best = (tot, res, hypo, w_ref)

    tot, res, hypo, w_ref = best
    out = {}
    for side, (seq, a) in res.items():
        for i, sl in zip(seq, a):
            if sl is None: continue
            slot, t = sl
            out[i] = QUAD[(arch, side, t)]*10 + slot
    return out, {"hypo": hypo, "cost": tot, "w_ref": w_ref, "n": len(out)}

def number_photo(n):
    d = SEG[n]; keep = d["keep"]
    up, lo = split_arches(keep)
    tsc = {**arch_types(keep, up), **arch_types(keep, lo)}
    fu, iu = number_arch(keep, up, tsc, "upper")
    fl, il = number_arch(keep, lo, tsc, "lower")
    return {**fu, **fl}, {"upper": iu, "lower": il}, tsc

FDI, INFO, TSC = {}, {}, {}
for n in SEG:
    FDI[n], INFO[n], TSC[n] = number_photo(n)
    up = sorted(v for v in FDI[n].values() if v//10 in (1,2,5,6))
    lo = sorted(v for v in FDI[n].values() if v//10 in (3,4,7,8))
    print(f"  {n[:40]:40s} atas {up}  bawah {lo}")

## 6. Uji akal sehat DP alignment

Empat lengkung sintetis dengan jawaban yang sudah diketahui. Ini menguji **implementasi**,
terpisah dari pertanyaan apakah SAM 3 mensegmentasi dengan benar.

| Kasus | Harapan |
|---|---|
| Permanen lengkap | 13,12,11,21,22,23 |
| **Lateral kiri tak tersegmentasi** | 13,11,21,22,23 — kaninus tetap **13**, tidak bergeser jadi 12 |
| **Diastema median** | 13,12,11,21,22,23 — celah tidak dianggap gigi hilang |
| Susu lengkap | 53,52,51,61,62,63 |
| Campuran (sentral permanen, sisanya susu) | 53,52,11,21,62,63 |

Perhatikan bedanya kasus kedua dengan "gigi dicabut lalu ruangnya menutup". Yang kita uji adalah
**gigi ada tapi gagal tersegmentasi**, sehingga ruangnya tetap ada — dan itulah mode kegagalan
yang benar-benar terjadi di pipeline kita.

In [ ]:
def _fake(widths, whs=None, gap=0.0, drop=(), extra_gap=None):
    """
    Bangun keep tiruan dari daftar lebar (kiri->kanan).
    drop      = indeks gigi yang TIDAK tersegmentasi (posisinya tetap dipakai -> ruang tersisa)
    extra_gap = {indeks: tambahan_jarak} untuk mensimulasikan diastema
    """
    whs = whs or [0.85]*len(widths)
    keep, x = [], 0.0
    for k, (w, wh) in enumerate(zip(widths, whs)):
        cx = x + w/2
        if k not in drop:
            keep.append((None, {"cx": cx, "cy": 100.0, "w": w, "h": w/wh,
                                "wh": wh, "area": w*w, "val": .9, "yellow": .05,
                                "tilt": 0., "y0": 0., "y1": 1., "score": .9}))
        x += w + gap + (extra_gap or {}).get(k, 0.0)
    return keep

def _check(label, keep, expect):
    idxs = list(range(len(keep)))
    tsc = arch_types(keep, idxs)
    got, info = number_arch(keep, idxs, tsc, "upper")
    seq = [got.get(i) for i in sorted(idxs, key=lambda i: keep[i][1]["cx"])]
    ok = seq == expect
    print(f"  {'OK  ' if ok else 'BEDA'} {label:36s} {seq}")
    if not ok: print(f"       diharapkan {' '*23}{expect}")
    print(f"       hipotesis={info['hypo']}  biaya={info['cost']:.2f}")
    return ok

# kaninus, lateral, sentral | sentral, lateral, kaninus
W_PERM = [7.6, 6.5, 8.5, 8.5, 6.5, 7.6]
W_PRIM = [7.0, 5.1, 6.5, 6.5, 5.1, 7.0]
WH_PERM, WH_PRIM = [0.80]*6, [1.10]*6

res = []
res.append(_check("permanen lengkap",
                  _fake(W_PERM, WH_PERM), [13,12,11,21,22,23]))
res.append(_check("lateral KIRI tak tersegmentasi",
                  _fake(W_PERM, WH_PERM, drop={1}), [13,11,21,22,23]))
res.append(_check("diastema median 3mm",
                  _fake(W_PERM, WH_PERM, extra_gap={2: 3.0}), [13,12,11,21,22,23]))
res.append(_check("susu lengkap",
                  _fake(W_PRIM, WH_PRIM), [53,52,51,61,62,63]))
res.append(_check("campuran: sentral permanen",
                  _fake([7.0, 5.1, 8.5, 8.5, 5.1, 7.0],
                        [1.10,1.10,0.80,0.80,1.10,1.10]), [53,52,11,21,62,63]))
print("\nSEMUA LULUS" if all(res) else "\nADA YANG BEDA — periksa RATIO / bobot biaya di CFG")

# --- uji ketahanan: seberapa besar spasi antar-gigi masih ditoleransi? ---
# Bukti posisi mengasumsikan gigi bersentuhan. Spasi umum (generalized spacing) menggeser
# semua gigi ke luar dan bisa dikira gigi hilang. Ini mengukur batas amannya.
print("\nketahanan terhadap spasi antar-gigi (harus tetap 13,12,11,21,22,23):")
for gp in (0.0, 0.3, 0.6, 1.0, 1.5, 2.0):
    keep = _fake(W_PERM, WH_PERM, gap=gp)
    idxs = list(range(len(keep)))
    got, _ = number_arch(keep, idxs, arch_types(keep, idxs), "upper")
    seq = [got.get(i) for i in sorted(idxs, key=lambda i: keep[i][1]["cx"])]
    print(f"  spasi {gp:.1f}mm -> {seq} {'OK' if seq==[13,12,11,21,22,23] else '<-- rusak'}")

## 7. Tinjau visual

Warna kontur: **hijau** = permanen atas, **merah** = permanen bawah, **biru** = susu atas,
**oranye** = susu bawah. Periksa tiga hal: apakah tiap kontur benar satu gigi, apakah pembagian
atas/bawah benar, dan apakah **11/21 (atau 51/61) benar-benar mengapit midline**.

In [ ]:
COL = {1:"#2ca02c", 2:"#2ca02c", 3:"#d62728", 4:"#d62728",
       5:"#1f77b4", 6:"#1f77b4", 7:"#ff7f0e", 8:"#ff7f0e"}

def show(n, ax=None):
    d = SEG[n]; keep, fdi = d["keep"], FDI[n]
    if ax is None: _, ax = plt.subplots(figsize=(7,5))
    ax.imshow(d["rgb"]); ax.axis("off")
    hy = INFO[n]["upper"]["hypo"] if INFO[n].get("upper") else "?"
    ax.set_title(f"{n[:40]}  (atas: {hy})", fontsize=8)
    for i, (m, g) in enumerate(keep):
        num = fdi.get(i)
        col = COL.get((num or 0)//10, "#999999")
        ax.contour(m, levels=[0.5], colors=col, linewidths=1.5)
        ax.text(g["cx"], g["cy"], str(num or "?"), color="white", fontsize=8, weight="bold",
                ha="center", va="center", bbox=dict(fc=col, ec="none", alpha=.85, pad=1.1))

ns = list(SEG); ncol = 3; nrow = int(np.ceil(len(ns)/ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(15, 3.7*nrow))
for a in np.ravel(axes): a.axis("off")
for k, n in enumerate(ns): show(n, np.ravel(axes)[k])
plt.suptitle("hijau/merah = permanen atas/bawah | biru/oranye = susu atas/bawah", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# ========= OVERRIDE MANUAL =========
# "nama file": {"drop":[idx,...], "fdi":{idx: nomor}}
OVERRIDE = {}

for n, ov in OVERRIDE.items():
    if n not in FDI: print("tidak ada:", n); continue
    for i in ov.get("drop", []): FDI[n].pop(i, None)
    FDI[n].update({int(k): int(v) for k, v in ov.get("fdi", {}).items()})
print("Override:", list(OVERRIDE) or "(belum ada)")

def idx_table(n):
    rows = [{"idx": i, "fdi": FDI[n].get(i), "cx": round(g["cx"]), "cy": round(g["cy"]),
             "w": round(g["w"],1), "wh": round(g["wh"],2),
             "p_susu": round(TSC[n].get(i, 0),2), "skor": round(g["score"],2)}
            for i, (m, g) in enumerate(SEG[n]["keep"])]
    return pd.DataFrame(rows).sort_values("cx")
print("\nContoh tabel indeks:"); print(idx_table(ns[0]).to_string(index=False))

## 8. Landmark & ekspor

In [ ]:
def landmarks(m, arch):
    ys, xs = np.nonzero(m)
    g = mask_geom(m)
    y0, y1 = ys.min(), ys.max(); H = max(y1-y0, 1)
    lo, hi = CFG["contact_band"]
    band = (ys >= y0+lo*H) & (ys <= y0+hi*H)
    if band.sum() < 10: band = np.ones_like(ys, bool)
    bx, by = xs[band], ys[band]
    L = np.array([bx[np.argmin(bx)], by[np.argmin(bx)]], float)
    R = np.array([bx[np.argmax(bx)], by[np.argmax(bx)]], float)
    f = CFG["incisal_frac"]
    sel = (ys >= y1-f*H) if arch == "upper" else (ys <= y0+f*H)
    if sel.sum() < 5: sel = np.ones_like(ys, bool)
    inc = np.array([np.median(xs[sel]), np.median(ys[sel])], float)
    return {**g, "contact_L": L, "contact_R": R, "incisal": inc}

LM = {}
for n in SEG:
    LM[n] = {}
    for i, (m, g) in enumerate(SEG[n]["keep"]):
        num = FDI[n].get(i)
        if not num: continue
        LM[n][num] = landmarks(m, "upper" if num//10 in (1,2,5,6) else "lower")

rows = []
for n in SEG:
    up = [v for v in FDI[n].values() if v//10 in (1,2,5,6)]
    lo = [v for v in FDI[n].values() if v//10 in (3,4,7,8)]
    prim = [v for v in FDI[n].values() if v//10 >= 5]
    iu = INFO[n].get("upper") or {}
    rows.append({"foto": n, "n_gigi": len(SEG[n]["keep"]), "n_atas": len(up), "n_bawah": len(lo),
                 "n_susu": len(prim), "dentisi_atas": iu.get("hypo"),
                 "biaya_align": round(iu.get("cost", np.nan), 2) if iu else np.nan,
                 "sentral_atas_lengkap": bool({11,21} <= set(up) or {51,61} <= set(up)),
                 "anterior_atas_6": len([v for v in up if v % 10 <= 3]) == 6})
SUM = pd.DataFrame(rows)
print(SUM.to_string(index=False))
SUM.to_csv(os.path.join(CFG["out_dir"], "summary_sam3.csv"), index=False)

with open(os.path.join(CFG["out_dir"], "fdi.json"), "w") as f:
    json.dump({n: {str(k): int(v) for k, v in FDI[n].items()} for n in FDI}, f, indent=1)
np.savez_compressed(os.path.join(CFG["out_dir"], "masks.npz"),
    **{f"{n}__{i}": SEG[n]["keep"][i][0] for n in SEG for i in range(len(SEG[n]["keep"]))})
print("\n->", CFG["out_dir"], sorted(os.listdir(CFG["out_dir"])))

In [ ]:
# Bangun ulang template rasio dari DATA SENDIRI (jalankan setelah penomoran diverifikasi).
# Lebih tepat daripada konstanta tekstbook: populasi cocok secara definisi.
ok_photos = SUM[SUM.anterior_atas_6 & SUM.sentral_atas_lengkap].foto.tolist()
print(f"{len(ok_photos)} foto dgn anterior atas lengkap & sentral terdeteksi")
if len(ok_photos) >= 5:
    acc = {}
    for n in ok_photos:
        lm = LM[n]
        base = [lm[t]["w"] for t in (11,21,51,61) if t in lm]
        if not base: continue
        ref = float(np.mean(base))
        for t, L in lm.items():
            if t//10 in (1,2,5,6) and t % 10 <= 4:
                acc.setdefault(t % 10, []).append(L["w"]/ref)
    print("\nrasio empiris (relatif sentral atas, median):")
    for slot in sorted(acc):
        v = np.array(acc[slot])
        print(f"  slot {slot}: {np.median(v):.3f}  (n={len(v)}, IQR "
              f"{np.percentile(v,25):.3f}-{np.percentile(v,75):.3f})")
    print("\nBandingkan dengan RATIO di Bagian 5. Kalau beda konsisten, ganti dgn angka ini.")
else:
    print("Belum cukup foto bersih — benahi segmentasi/penomoran dulu.")

## Batas & langkah berikutnya

- **Rasio lebar dari tekstbook** (Wheeler/Ash) berbasis populasi Kaukasia. Sel terakhir membangun
  ulang template dari datamu sendiri — pakai itu begitu ada ≥5 foto yang penomorannya terverifikasi.
- **Klasifikasi susu vs permanen belum tervalidasi** terhadap penilaian dokter. Kolom `p_susu` di
  tabel indeks menunjukkan skornya; minta drg. Laura memeriksa foto dentisi campuran secara khusus.
- **Kriteria w/h berlaku untuk insisivus**, tidak untuk kaninus/molar. DP alignment sudah
  mengompensasi lewat rasio lebar, tapi tebakan awal tipe pada gigi posterior memang lemah.
- Setelah penomoran bersih: hitung ulang metrik kerapihan (notebook segmentasi SAM 2 Bagian 6)
  dan **uji ulang korelasinya dengan penilaian AC** — pada run sebelumnya `LII_norm` vs AC hanya
  rho = +0.085, dan hipotesis utamanya adalah segmentasi yang rusak.